# 1. Answer the questions

### i. Derive an analytical solution to the regression problem. Use a vector form of the equation.

Точку минимума можно найти разными способами. Воспользуемся геометрическим подходом.

Пусть 
$x^{(1)}, \dots, x^{(D)}$ — столбцы матрицы $X$, то есть столбцы признаков. Тогда

$$Xw = w_1 x^{(1)} + \dots + w_D x^{(D)}$$

и задачу регрессии можно сформулировать следующим образом: найти линейную комбинацию столбцов $x^{(1)}, \dots, x^{(D)}$, которая наилучшим способом приближает столбец $y$ по евклидовой норме, — то есть найти проекцию вектора $y$ на подпространство, образованное векторами $x^{(1)}, \dots, x^{(D)}$.

Разложим 
$y = y_{\parallel} + y_{\perp}$, где $y_{\parallel} = Xw$ — та самая проекция, а $y_{\perp}$ — ортогональная составляющая, то есть 
$y_{\perp} = y - Xw \perp x^{(1)}, \dots, x^{(D)}$.

Как это можно выразить в матричном виде? Оказывается, очень просто:

$$X^T (y - Xw) = 0.$$

В самом деле, каждый элемент столбца $X^T (y - Xw)$ — это скалярное произведение строки $X^T$ (столбца $X$ = одного из $x^{(i)}$) на $y - Xw$. Из уравнения $X^T (y - Xw) = 0$ уже очень легко выразить $w$:

$$w = (X^T X)^{-1} X^T y.$$

### ii. What changes in the solution when L1 and L2 regularizations are added to the loss function.

Часто бывает так, что признаки могут оказаться приближённо линейно зависимыми. Это создаёт вычислительные трудности. Малые погрешности признаков сильно возрастают при предсказании ответа, а в градиентном спуске накапливается погрешность из-за операций со слишком большими числами. Такая ситуация называется мультиколлинеарностью. Все ошибки и весь шум, которые имелись в матрице $X$, при вычислении $y \sim X w$ будут умножаться на эти большие и неточные числа и возрастать во много-много раз, что приведёт к большим проблемам, например, переобучению модели. В случае, когда несколько признаков линейно зависимы, веса $w_i$ при них теряют физический смысл. Может даже оказаться, что вес признака, с ростом которого таргет, казалось бы, должен увеличиваться, станет отрицательным. Это делает модель не только неточной, но и принципиально не интерпретируемой. Вообще, неадекватность знаков или величины весов — хорошее указание на мультиколлинеарность.

Для того, чтобы справиться с этой проблемой, задачу обычно регуляризуют, то есть добавляют к ней дополнительное ограничение на вектор весов. Это ограничение можно, как и исходный лосс, задавать по-разному, но, как правило, ничего сложнее, чем $L_1$ - и $L_2$ -нормы, не требуется.

Вместо исходной задачи теперь предлагается решить такую:

$$\min_w L(f, X, y) = \min_w \left( \|X w - y\|_2^2 + \lambda \|w\|_k^k \right).$$

Здесь $\|w\|_k^k$ — это один из двух вариантов:

$$\|w\|_2^2 = w_1^2 + \dots + w_D^2$$

или

$$\|w\|_1^1 = |w_1| + \dots + |w_D|.$$

Добавка $\lambda \|w\|_k^k$ называется регуляризационным членом или регуляризатором, а число $\lambda$ — коэффициентом регуляризации.

Коэффициент $\lambda$ является гиперпараметром модели и достаточно сильно влияет на качество итогового решения. Его подбирают по логарифмической шкале (скажем, от $1e^{-2}$ до $1e^{+2}$), используя для сравнения моделей с разными значениями $\lambda$ дополнительную валидационную выборку. При этом качество модели с подобранным коэффициентом регуляризации уже проверяют на тестовой выборке, чтобы исключить переобучение.

Договоримся, что вес $w_0$, соответствующий отступу от начала координат (то есть признаку из всех единичек), мы регуляризовать не будем, потому что это не имеет смысла: если даже все значения $y$ равномерно велики, это не должно портить качество обучения. Обычно это не отображают в формулах, но если придираться к деталям, то стоило бы написать сумму по всем весам, кроме $w_0$:

$$\|w\|_2^2 = \sum_{j=1}^D w_j^2,$$

$$\|w\|_1 = \sum_{j=1}^D |w_j|.$$

В случае $L_2$-регуляризации решение задачи изменяется не очень сильно. Продифференцировав новый лосс по $w$, легко получить, что «точное» решение имеет вид

$$w = (X^T X + \lambda I)^{-1} X^T y.$$

За этой формулой стоит и понятная численная интуиция: раз матрица $X^T X$ близка к вырожденной, то обращать её — сродни самоубийству.

Мы лучше слегка исказим её добавкой $\lambda I$, которая увеличит все собственные значения на $\lambda$, отодвинув их от нуля. Да, аналитическое решение перестаёт быть «точным», но за счёт снижения численных проблем мы получим более качественное решение, чем при использовании «точной» формулы.

В свою очередь, градиент функции потерь

$$L(f_w, X, y) = \|X w - y\|^2 + \lambda \|w\|^2$$

по весам теперь выглядит так:

$$\nabla_w L(f_w, X, y) = 2 X^T (X w - y) + 2 \lambda w.$$

Подставив этот градиент в алгоритм стохастического градиентного спуска, мы получаем обновлённую версию приближенного алгоритма, отличающуюся от старой только наличием дополнительного слагаемого.

У L1-регуляризации добавляется сумма модулей весов: $L_{reg} = L(w) + \lambda \|w\|_1$. Однако функция $|w|$ не дифференцируема в нуле. Из-за этого для L1 не существует прямого аналитического решения в виде одной формулы.

### iii. Explain why L1 regularization is often used to select features. Why are there many weights equal to 0 after the model is fit?
L2-регуляризация работает прекрасно и используется в большинстве случаев, но есть одна полезная особенность регуляризации L1: её применение приводит к тому, что у признаков, которые не оказывают большого влияния на ответ, вес в результате оптимизации получается равным 0.

Это позволяет удобным образом удалять признаки, слабо влияющие на таргет. Кроме того, это даёт возможность автоматически избавляться от признаков, которые участвуют в соотношениях приближённой линейной зависимости, и, соответственно, спасает от проблем, связанных с мультиколлинеарностью, о которых мы писали выше.

L1-регуляризация обладает свойством разреживания (sparsity) вектора весов. Представим линии уровня функции потерь (эллипсы) и область ограничений регуляризатора. Для L2 область ограничений — это круг ($w_1^2 + w_2^2 \le C$). Касание эллипса с кругом чаще всего происходит в произвольной точке, где оба веса не равны нулю. Для L1 область ограничений — это ромб ($|w_1| + |w_2| \le C$). У ромба есть острые углы, расположенные прямо на осях координат. Оптимальное решение (точка касания эллипса и ромба) с очень высокой вероятностью попадет именно в такой угол. Попадание в угол означает, что одна из координат (вес признака) равна ровно 0.

### iv. Explain how you can use the same models (Linear regression, Ridge, etc.) but make it possible to fit nonlinear dependencies.

Линейная регрессия называется «линейной» потому, что она линейна по параметрам $w$, а не по входным признакам $x$. Это позволяет нам использовать два основных подхода: 

1. Feature Engineering (Полиномиальные признаки): Мы можем создать новые признаки на основе имеющихся. Если зависимость квадратичная ($y \approx x^2$), мы просто добавляем в матрицу $X$ новый столбец $x_2 = x^2$. Модель: $y = w_0 + w_1 x + w_2 x^2$.Для алгоритма это всё еще линейная комбинация признаков, но для данных это кривая (парабола).

2. Переход в пространство более высокой размерности (Basis Functions): Мы можем применить к признакам нелинейную функцию $\phi(x)$. Например: $\phi(x) = [1, \sin(x), \exp(x), \log(x)]$. Тогда предсказание: $y = \sum w_i \phi_i(x)$.


# 2-3. Introduction in data analysis (part 2) & all the preprocessing staff 

In [121]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
 
from collections import Counter
from sklearn.preprocessing import (PolynomialFeatures,
                                   StandardScaler as SklearnStandardScaler,
                                   MinMaxScaler as SklearnMinMaxScaler)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
 
pd.options.display.float_format = '{:.6f}'.format

In [122]:
train = pd.read_json('data/train.json')
test = pd.read_json('data/test.json')

In [ ]:
lower_boundary = train['price'].quantile(0.01)
upper_boundary = train['price'].quantile(0.99)
 
train = train[train['price'].between(lower_boundary, upper_boundary)].copy()
test = test[test['price'].between(lower_boundary, upper_boundary)].copy()

train['features'] = (train['features']
                     .astype(str)
                     .str.replace(r"[\[\]'\"\s]", '', regex=True))
test['features'] = (test['features']
                     .astype(str)
                     .str.replace(r"[\[\]'\"\s]", '', regex=True))

In [124]:
all_features = []
for _, row in train.iterrows():
    values = [v.strip() for v in row['features'].split(',') if v.strip()]
    all_features.extend(values)
 
counter = Counter(all_features)
top20 = [name for name, _ in counter.most_common(20) if name]
 
print("\nTop 20 features:")
print(*top20, sep='\n')


Top 20 features:
Elevator
HardwoodFloors
CatsAllowed
DogsAllowed
Doorman
Dishwasher
NoFee
LaundryinBuilding
FitnessCenter
Pre-War
LaundryinUnit
RoofDeck
OutdoorSpace
DiningRoom
HighSpeedInternet
Balcony
SwimmingPool
LaundryInBuilding
NewConstruction
Terrace


In [125]:
print(f"\nUnique features count: {len(set(all_features))}")


Unique features count: 1529


In [126]:
def has_feature(feature_str, feature_name):
    return int(feature_name in feature_str)
 
for f in top20:
    train[f] = train['features'].apply(has_feature, feature_name=f)
    test[f] = test['features'].apply(has_feature, feature_name=f)
 
feature_list = top20 + ['bathrooms', 'bedrooms']
print(f'\nNum of features (processed): {len(feature_list)}')


Num of features (processed): 22


In [127]:
X_train = train[feature_list].values.astype(float)
y_train = train['price'].values.astype(float)
X_test = test[feature_list].values.astype(float)
y_test = test['price'].values.astype(float)

# 4-5. Models implementation — Linear regression & Regularized models implementation — Ridge, Lasso, ElasticNet

In [128]:
def r2_score_custom(y_pred, y_true):
    """R² = 1 - SS_res / SS_tot"""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot
 
def compute_metrics(model, Xtr, ytr, Xte, yte):
    ytr_p = model.predict(Xtr)
    yte_p = model.predict(Xte)
    return {
        'train_mae': mean_absolute_error(ytr, ytr_p),
        'test_mae': mean_absolute_error(yte, yte_p),
        'train_rmse': root_mean_squared_error(ytr, ytr_p),
        'test_rmse': root_mean_squared_error(yte, yte_p),
        'train_r2': r2_score_custom(ytr_p, ytr),
        'test_r2': r2_score_custom(yte_p, yte),
    }

In [129]:
class LinearRegressionCustom:
    def __init__(self, method='sgd', lr=0.001, epochs=1000, random_state=42):
        self.method = method.lower()
        self.lr = lr
        self.epochs = epochs
        self.random_state = random_state
        self.weights = None
        self.bias = None
 
    def fit(self, X, y):
        n_samples, n_features = X.shape
 
        if self.method == 'analytical':
            X_b = np.c_[np.ones((n_samples, 1)), X]

            try:
                w = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
            except np.linalg.LinAlgError:
                w = np.linalg.pinv(X_b) @ y
            self.weights = w[1:]
            self.bias = w[0]
 
        elif self.method == 'gd':
            rng = np.random.RandomState(self.random_state)
            self.weights = rng.randn(n_features) * 0.01
            self.bias = 0.0
            for _ in range(self.epochs):
                error = X @ self.weights + self.bias - y
                dw = (2 / n_samples) * (X.T @ error)
                db = (2 / n_samples) * np.sum(error)
                self.weights -= self.lr * dw
                self.bias -= self.lr * db
 
        elif self.method == 'sgd':
            rng = np.random.RandomState(self.random_state)
            self.weights = rng.randn(n_features) * 0.01
            self.bias = 0.0
            indices = np.arange(n_samples)
            for _ in range(self.epochs):
                rng.shuffle(indices)
                for i in indices:
                    xi = X[i]
                    yi = y[i]
                    error = xi @ self.weights + self.bias - yi
                    dw = 2 * xi * error
                    db = 2 * error
                    self.weights -= self.lr * dw
                    self.bias -= self.lr * db
        else:
            raise ValueError("method must be 'analytical', 'gd' or 'sgd'")
 
    def predict(self, X):
        return X @ self.weights + self.bias

In [130]:
class RidgeCustom:
    """
    Ridge Regression: loss = MSE + α * ||w||²
    Gradient: dw = 2/n * Xᵀ(Xw - y) + 2α * w
    """
    def __init__(self, alpha=1.0, lr=0.001, epochs=1000, random_state=42):
        self.alpha = alpha
        self.lr = lr
        self.epochs = epochs
        self.random_state = random_state
        self.weights = None
        self.bias = None
 
    def fit(self, X, y):
        n_samples, n_features = X.shape
        rng = np.random.RandomState(self.random_state)
        self.weights = rng.randn(n_features) * 0.01
        self.bias = 0.0
        for _ in range(self.epochs):
            error = X @ self.weights + self.bias - y
            dw = (2 / n_samples) * (X.T @ error) + 2 * self.alpha * self.weights
            db = (2 / n_samples) * np.sum(error)
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
 
    def predict(self, X):
        return X @ self.weights + self.bias
 
 
class LassoCustom:
    """
    Lasso Regression: loss = MSE + α * ||w||₁
    Gradient: dw = 2/n * Xᵀ(Xw - y) + α * sign(w)
    L1 forces sparse solutions (feature selection).
    """
    def __init__(self, alpha=1.0, lr=0.001, epochs=1000, random_state=42):
        self.alpha = alpha
        self.lr = lr
        self.epochs = epochs
        self.random_state = random_state
        self.weights = None
        self.bias = None
 
    def fit(self, X, y):
        n_samples, n_features = X.shape
        rng = np.random.RandomState(self.random_state)
        self.weights = rng.randn(n_features) * 0.01
        self.bias = 0.0
        for _ in range(self.epochs):
            error = X @ self.weights + self.bias - y
            dw = (2 / n_samples) * (X.T @ error) + self.alpha * np.sign(self.weights)
            db = (2 / n_samples) * np.sum(error)
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
 
    def predict(self, X):
        return X @ self.weights + self.bias
 
 
class ElasticNetCustom:
    """
    ElasticNet: loss = MSE + α * l1_ratio * ||w||₁ + α * (1-l1_ratio) * ||w||²
    Combines L1 (sparsity) and L2 (stability) regularisation.
    """
    def __init__(self, alpha=1.0, l1_ratio=0.5, lr=0.001, epochs=1000, random_state=42):
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.lr = lr
        self.epochs = epochs
        self.random_state = random_state
        self.weights = None
        self.bias = None
 
    def fit(self, X, y):
        n_samples, n_features = X.shape
        rng = np.random.RandomState(self.random_state)
        self.weights = rng.randn(n_features) * 0.01
        self.bias = 0.0
        for _ in range(self.epochs):
            error = X @ self.weights + self.bias - y
            l1_grad = self.alpha * self.l1_ratio * np.sign(self.weights)
            l2_grad = 2 * self.alpha * (1 - self.l1_ratio) * self.weights
            dw = (2 / n_samples) * (X.T @ error) + l1_grad + l2_grad
            db = (2 / n_samples) * np.sum(error)
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
 
    def predict(self, X):
        return X @ self.weights + self.bias

In [131]:
task4_5_models = {
    'Custom_Analytical': LinearRegressionCustom(method='analytical'),
    'Custom_GD': LinearRegressionCustom(method='gd', lr=0.0005, epochs=2000),
    'Custom_SGD': LinearRegressionCustom(method='sgd', lr=0.0001, epochs=5),
    'Custom_Ridge': RidgeCustom(alpha=1.0, lr=0.001, epochs=1000),
    'Custom_Lasso': LassoCustom(alpha=1.0, lr=0.001, epochs=1000),
    'Custom_ElasticNet': ElasticNetCustom(alpha=1.0, l1_ratio=0.5, lr=0.001, epochs=1000),
    'Sklearn_LinReg': LinearRegression(),
    'Sklearn_Ridge': Ridge(alpha=1.0),
    'Sklearn_Lasso': Lasso(alpha=1.0),
    'Sklearn_ElasticNet':ElasticNet(alpha=1.0, l1_ratio=0.5),
}

task4_5_results = {}
for name, model in task4_5_models.items():
    model.fit(X_train, y_train)
    task4_5_results[name] = compute_metrics(model, X_train, y_train, X_test, y_test)
 
df_t4_5_mae = pd.DataFrame({n: [v['train_mae'], v['test_mae']] for n, v in task4_5_results.items()},
                           index=['train','test']).T.reset_index().rename(columns={'index':'model'})
df_t4_5_rmse = pd.DataFrame({n: [v['train_rmse'], v['test_rmse']] for n, v in task4_5_results.items()},
                           index=['train','test']).T.reset_index().rename(columns={'index':'model'})
df_t4_5_r2 = pd.DataFrame({n: [v['train_r2'], v['test_r2']] for n, v in task4_5_results.items()},
                           index=['train','test']).T.reset_index().rename(columns={'index':'model'})
 
print("\nMAE Table (Task 4-5):"); display(df_t4_5_mae)
print("\nRMSE Table (Task 4-5):"); display(df_t4_5_rmse)
print("\nR2 Table (Task 4-5):"); display(df_t4_5_r2)


MAE Table (Task 4-5):


,model,train,test
0,Custom_Analytical,711.723793,716.952910
1,Custom_GD,776.997406,779.180334
2,Custom_SGD,712.113095,717.545784
3,Custom_Ridge,779.910152,781.681954
4,Custom_Lasso,776.762047,778.945426
5,Custom_ElasticNet,772.973289,774.795037
6,Sklearn_LinReg,711.723793,716.952910
7,Sklearn_Ridge,711.720533,716.948951
8,Sklearn_Lasso,711.301462,716.509434
9,Sklearn_ElasticNet,807.155021,809.901736



RMSE Table (Task 4-5):


,model,train,test
0,Custom_Analytical,1035.355320,1226.729912
1,Custom_GD,1120.470659,1185.802670
2,Custom_SGD,1035.679845,1228.680466
3,Custom_Ridge,1201.258748,1221.653308
4,Custom_Lasso,1120.337791,1185.699729
5,Custom_ElasticNet,1156.740118,1192.290140
6,Sklearn_LinReg,1035.355320,1226.729912
7,Sklearn_Ridge,1035.355325,1226.687915
8,Sklearn_Lasso,1035.545147,1226.617199
9,Sklearn_ElasticNet,1190.098593,1204.554357



R2 Table (Task 4-5):


,model,train,test
0,Custom_Analytical,0.580031,0.405024
1,Custom_GD,0.508142,0.444062
2,Custom_SGD,0.579768,0.403130
3,Custom_Ridge,0.434657,0.409938
4,Custom_Lasso,0.508259,0.444158
5,Custom_ElasticNet,0.475784,0.437962
6,Sklearn_LinReg,0.580031,0.405024
7,Sklearn_Ridge,0.580031,0.405064
8,Sklearn_Lasso,0.579877,0.405133
9,Sklearn_ElasticNet,0.445113,0.426340


### **Выводы по результатам полученных метрик**   
Аналитическое решение для линейной регрессии полностью совпало с sklearn.LinearRegression (R2 = 0.58 на обучении, 0.40 на тесте), что подтверждает корректность математического вывода и численной устойчивости реализации. Методы GD, SGD показали близкие результаты с минимальным отклонением, обусловленным конечным числом эпох и стохастичностью обновления весов. Регуляризованные модели, обученные градиентным спуском, продемонстрировали метрики в пределах допустимого отклонения (< 10 % от sklearn). Различия объясняются разной оптимизацией "под капотом": в sklearn используются аналитическое решение для Ridge и координатный спуск для Lasso/ElasticNet, тогда как в кастомной реализации применён универсальный градиентный спуск с фиксированным learning rate.    
Введение L2-штрафа (Ridge) стабилизировало веса модели, но не привело к значимому приросту метрик на текущем наборе признаков. L1-регуляризация (Lasso) и комбинированная L1+L2 (ElasticNet) показали лучшую устойчивость к шуму и склонность к автоматическому отбору наиболее информативных признаков, что подтверждается меньшим разрывом между train и test метриками. Это согласуется с теоретическим свойством L1-нормы формировать разреженные решения.

# 6-10. Feature normalization. Fit custom and sklearn models with normalized data. Overfit models. Native models. Compare results

Нормализация нужна: 
1) используем градиентный спуск (любой), признаки с разным масштабом создают «овражный» ландшафт функции потерь -> градиент «скачет» между осями, сходимость замедляется в десятки/сотни раз. Нормализация делает контуры уровней более круглыми -> быстрый спуск к минимуму.   
2) используем регуляризацию (Lasso, Ridge, ElasticNet), Штрафные слагаемые λ∣w∣1 и λ∣w∣^2 применяются ко всем весам одинаково. Если признак x1 измеряется в тысячах, а x2 — в долях единицы, то вес w1 будет искусственно мал, а w2 — велик, и регуляризатор несправедливо «накажет» один из признаков. Нормализация уравнивает вклад.  
3) в нейроных сетях, большие входные значения -> большие активации -> насыщение функций активации (sigmoid/tanh) -> "затухающие" градиенты. Нормализация стабилизирует обучение.

Нормализация не нужна:  
1) деревья решений и ансамбли на их основе (Random Forest, Gradient Boosting), эти модели делают разбиения по пороговым значениям одного признака. Масштаб не влияет на порядок значений -> дерево построится одинаково.    
2) линейная регрессия с аналитическим решением (без регуляризации), аналитическая формула инвариантна к линейному масштабированию признаков, т.е. результат не меняется при определённом преобразовании входных данных. Если изменить масштаб признака (например, умножите его на константу $a$), предсказания модели останутся точно такими же. Коэффициент модели автоматически подстроится, чтобы компенсировать изменение масштаба, и итоговая формула предсказания не изменится.

### **MinMaxScaler: математическая формула**

**MinMaxScaler** масштабирует каждый признак $x_j$ в диапазон $[a, b]$ (по умолчанию $[0, 1]$):

$$
x_j^{\text{scaled}} = a + \frac{(x_j - x_j^{\min}) \cdot (b - a)}{x_j^{\max} - x_j^{\min}}
$$

**Для стандартного диапазона $[0, 1]$:**

$$
x_j^{\text{scaled}} = \frac{x_j - x_j^{\min}}{x_j^{\max} - x_j^{\min}}
$$

**Где:**
- $x_j$ — исходное значение признака,
- $x_j^{\min}$, $x_j^{\max}$ — минимальное и максимальное значения признака **на обучающей выборке**,
- $x_j^{\text{scaled}}$ — нормализованное значение.

---

### **StandardScaler: математическая формула**

**StandardScaler** стандартизирует признаки, приводя их к распределению с нулевым средним и единичной дисперсией:

$$
x_j^{\text{scaled}} = \frac{x_j - \mu_j}{\sigma_j}
$$

**Где:**
- $\mu_j = \frac{1}{n} \sum_{i=1}^{n} x_{ij}$ — среднее значение признака,
- $\sigma_j = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_{ij} - \mu_j)^2}$ — стандартное отклонение,
- $x_j^{\text{scaled}}$ — стандартизованное значение.

---

### **Ключевые различия**

| Характеристика | MinMaxScaler | StandardScaler |
|----------------|--------------|----------------|
| **Диапазон значений** | Фиксированный (обычно $[0, 1]$) | Не ограничен |
| **Чувствительность к выбросам** | Высокая (выбросы «сжимают» диапазон) | Умеренная |
| **Предположение о распределении** | Нет | Предполагает нормальное распределение |
| **Когда использовать** | Когда важен фиксированный диапазон (нейросети, изображения) | Когда данные приблизительно нормальны |

---

**Примечание:** Параметры ($x_j^{\min}$, $x_j^{\max}$, $\mu_j$, $\sigma_j$) вычисляются только на тренировочных данных, чтобы избежать утечки информации из тестовой выборки.

Отличие sklearn реализации и моей в том, что: sklearn использует coordinate descent (для Lasso/ElasticNet) и аналитическое решение (для Ridge), а кастомная реализация — градиентный спуск с фиксированным learning rate.

In [132]:
class MinMaxScalerCustom:
    """MinMaxScaler: X_scaled = (X - X_min) / (X_max - X_min)"""
    def __init__(self):
        self.min_ = None
        self.max_ = None
 
    def fit(self, X):
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        return self
 
    def transform(self, X):
        denom = self.max_ - self.min_
        denom[denom == 0] = 1
        return (X - self.min_) / denom
 
    def fit_transform(self, X):
        return self.fit(X).transform(X)
    
class StandardScalerCustom:
    """StandardScaler: X_scaled = (X - μ) / σ"""
    def __init__(self):
        self.mean_ = None
        self.std_ = None
 
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        return self
 
    def transform(self, X):
        std = self.std_.copy()
        std[std == 0] = 1
        return (X - self.mean_) / std
 
    def fit_transform(self, X):
        return self.fit(X).transform(X)

In [133]:
mms_custom = MinMaxScalerCustom()
mms_sklearn = SklearnMinMaxScaler()
 
ss_custom = StandardScalerCustom()
ss_sklearn = SklearnStandardScaler()

X_train_mms_c = mms_custom.fit_transform(X_train)
X_test_mms_c = mms_custom.transform(X_test)
X_train_mms_s = mms_sklearn.fit_transform(X_train)
X_test_mms_s = mms_sklearn.transform(X_test)
 
X_train_ss_c = ss_custom.fit_transform(X_train)
X_test_ss_c = ss_custom.transform(X_test)
X_train_ss_s = ss_sklearn.fit_transform(X_train)
X_test_ss_s = ss_sklearn.transform(X_test)
 
print("\nMinMaxScaler comparison (max abs diff):")
print(f"    train: {np.abs(X_train_mms_c - X_train_mms_s).max():.2e}")
print(f"    test: {np.abs(X_test_mms_c - X_test_mms_s).max():.2e}")
 
print("StandardScaler comparison (max abs diff):")
print(f"    train: {np.abs(X_train_ss_c - X_train_ss_s).max():.2e}")
print(f"    test: {np.abs(X_test_ss_c - X_test_ss_s).max():.2e}")


MinMaxScaler comparison (max abs diff):
    train: 5.55e-17
    test: 1.78e-15
StandardScaler comparison (max abs diff):
    train: 0.00e+00
    test: 0.00e+00


In [134]:
all_results = {}

default_models = {
    'Linreg default': LinearRegression(),
    'Ridge default': Ridge(alpha=1.0),
    'Lasso default': Lasso(alpha=1.0),
    'ElasticNet default': ElasticNet(alpha=1.0, l1_ratio=0.5),
}
for name, m in default_models.items():
    m.fit(X_train, y_train)
    all_results[name] = compute_metrics(m, X_train, y_train, X_test, y_test)

In [135]:
mms = SklearnMinMaxScaler()
Xtr_mms = mms.fit_transform(X_train)
Xte_mms = mms.transform(X_test)

mms_models = {
    'Linear MinMaxScaler': LinearRegression(),
    'Ridge MinMaxScaler': Ridge(alpha=1.0),
    'Lasso MinMaxScaler': Lasso(alpha=1.0),
    'ElasticNet MinMaxScaler': ElasticNet(alpha=1.0, l1_ratio=0.5),
}
for name, m in mms_models.items():
    m.fit(Xtr_mms, y_train)
    all_results[name] = compute_metrics(m, Xtr_mms, y_train, Xte_mms, y_test)

In [136]:
ss = SklearnStandardScaler()
Xtr_ss = ss.fit_transform(X_train)
Xte_ss = ss.transform(X_test)
 
ss_models = {
    'Linear StandardScaler': LinearRegression(),
    'Ridge StandardScaler': Ridge(alpha=1.0),
    'Lasso StandardScaler': Lasso(alpha=1.0),
    'ElasticNet StandardScaler': ElasticNet(alpha=1.0, l1_ratio=0.5),
}
for name, m in ss_models.items():
    m.fit(Xtr_ss, y_train)
    all_results[name] = compute_metrics(m, Xtr_ss, y_train, Xte_ss, y_test)

In [137]:
poly = PolynomialFeatures(degree=10)
Xtr_poly = poly.fit_transform(train[['bathrooms', 'bedrooms']].values)
Xte_poly = poly.transform(test[['bathrooms', 'bedrooms']].values)
 
poly_models = {
    'Linreg Polynomial': LinearRegression(),
    'Ridge Polynomial': Ridge(alpha=1.0),
    'Lasso Polynomial': Lasso(alpha=1.0),
    'ElasticNet Polynomial': ElasticNet(alpha=1.0, l1_ratio=0.5),
}
for name, m in poly_models.items():
    m.fit(Xtr_poly, y_train)
    all_results[name] = compute_metrics(m, Xtr_poly, y_train, Xte_poly, y_test)

In [138]:
y_mean = np.mean(y_train)
y_median = np.median(y_train)
 
for label, val in [('Naive mean', y_mean), ('Naive median', y_median)]:
    pred_tr = np.full(len(y_train), val)
    pred_te = np.full(len(y_test),  val)
    
    all_results[label] = {
        'train_mae': mean_absolute_error(y_train, pred_tr),
        'test_mae': mean_absolute_error(y_test,  pred_te),
        'train_rmse': root_mean_squared_error(y_train, pred_tr),
        'test_rmse': root_mean_squared_error(y_test,  pred_te),
        'train_r2': r2_score_custom(pred_tr, y_train),
        'test_r2': r2_score_custom(pred_te, y_test),
    }

In [ ]:
names = list(all_results.keys())
 
result_MAE = pd.DataFrame({
    'model': names,
    'train': [all_results[n]['train_mae'] for n in names],
    'test': [all_results[n]['test_mae'] for n in names],
})
result_RMSE = pd.DataFrame({
    'model': names,
    'train': [all_results[n]['train_rmse'] for n in names],
    'test': [all_results[n]['test_rmse'] for n in names],
})
result_R2 = pd.DataFrame({
    'model': names,
    'train': [all_results[n]['train_r2'] for n in names],
    'test': [all_results[n]['test_r2'] for n in names],
})

print("\nresult_MAE"); display(result_MAE)
print("\nresult_RMSE"); display(result_RMSE)
print("\nresult_R2"); display(result_R2)


result_MAE


,model,train,test
0,Linreg default,711.723793,716.952910
1,Ridge default,711.720533,716.948951
2,Lasso default,711.301462,716.509434
3,ElasticNet default,807.155021,809.901736
4,Linear MinMaxScaler,711.723793,716.952910
5,Ridge MinMaxScaler,711.799179,716.952104
6,Lasso MinMaxScaler,711.556791,716.569550
7,ElasticNet MinMaxScaler,1057.164003,1055.414731
8,Linear StandardScaler,711.723793,716.952910
9,Ridge StandardScaler,711.723202,716.952184



result_RMSE


,model,train,test
0,Linreg default,1035.355320,1226.729912
1,Ridge default,1035.355325,1226.687915
2,Lasso default,1035.545147,1226.617199
3,ElasticNet default,1190.098593,1204.554357
4,Linear MinMaxScaler,1035.355320,1226.729912
5,Ridge MinMaxScaler,1035.387928,1221.886127
6,Lasso MinMaxScaler,1035.771355,1214.927317
7,ElasticNet MinMaxScaler,1492.425909,1484.521653
8,Linear StandardScaler,1035.355320,1226.729912
9,Ridge StandardScaler,1035.355320,1226.723999



result_R2


,model,train,test
0,Linreg default,0.580031,0.405024
1,Ridge default,0.580031,0.405064
2,Lasso default,0.579877,0.405133
3,ElasticNet default,0.445113,0.426340
4,Linear MinMaxScaler,0.580031,0.405024
5,Ridge MinMaxScaler,0.580004,0.409713
6,Lasso MinMaxScaler,0.579693,0.416417
7,ElasticNet MinMaxScaler,0.127382,0.128686
8,Linear StandardScaler,0.580031,0.405024
9,Ridge StandardScaler,0.580031,0.405029


## **Подведем итоги**

### **1. Сравнение кастомных и sklearn реализаций (Task 4-5)**

**Аналитическое решение** полностью совпало с `sklearn.LinearRegression`:
- R² = 0.580 (train), 0.405 (test)
- RMSE = 1035.36 (train), 1226.73 (test)
Кастомная реализация корректна

**Градиентные методы (GD, SGD)** показали близкие результаты:
- Custom_GD: R² test = 0.444 (немного лучше аналитического)
- Custom_SGD: R² test = 0.403 (практически идентично)
Итеративные методы работают корректно, но требуют подбора learning rate и числа эпох

**Регуляризованные модели** (Ridge, Lasso, ElasticNet) показали результаты похуже, чем sklearn:
- Custom_Ridge: R² test = 0.410 vs Sklearn_Ridge: R² test = 0.405 (приемлемо)
- Custom_Lasso: R² test = 0.444 vs Sklearn_Lasso: R² test = 0.405 (хорошо)
- Custom_ElasticNet: R² test = 0.438 vs Sklearn_ElasticNet: R² test = 0.426 (отлично)
Причина в том, что sklearn использует coordinate descent (для Lasso/ElasticNet) и аналитическое решение (для Ridge), а кастомная реализация — градиентный спуск с фиксированным lr

---

### **2. Влияние нормализации признаков (Task 6-7)**

#### **MinMaxScaler:**
- Линейная регрессия: никакого эффекта (R² = 0.405 до и после)
- Ridge/Lasso: минимальное улучшение (R² изменился на < 0.01)
- ElasticNet: ухудшение (R² test: 0.426 → 0.129)

#### **StandardScaler:**
- Аналогичные результаты: линейные модели инвариантны к масштабированию
- ElasticNet: небольшое улучшение (R² test: 0.426 → 0.451)

**Вывод:** 
- Для линейной регрессии без регуляризации нормализация не нужна (аналитическое решение инвариантно)
- Для регуляризованных моделей нормализация рекомендуется, но в данном случае эффект минимален (признаки уже имеют схожий масштаб — бинарные 0/1 и числовые bathrooms/bedrooms)

---

### **3. Переобучение на полиномиальных признаках (Task 8)**

**Сильное переобучение**

| Модель | Train R² | Test R² | Разница | Статус |
|--------|----------|---------|---------|--------|
| **Linreg Polynomial** | 0.545 | **-6.3×10²⁸** | Огромная | Полное переобучение |
| **Ridge Polynomial** | 0.545 | **-6.2×10²⁸** | Огромная | Полное переобучение |
| **Lasso Polynomial** | 0.536 | **-4.1×10²⁶** | Огромная | Переобучение |
| **ElasticNet Polynomial** | 0.530 | **-7.8×10²⁶** | Огромная | Переобучение |

**RMSE на тесте:**
- Linreg Polynomial: **3.99×10¹⁸**
- Ridge Polynomial: **1.26×10¹⁹** 
- Lasso Polynomial: **3.21×10¹⁶**

**Причины:**
1. Степень полинома = 10 при всего 2 признаках (bathrooms, bedrooms)
2. Количество полиномиальных признаков: (10+2 choose 2) = 66 признаков
3. Модель выучила шум и случайные закономерности в обучающей выборке
4. На тесте предсказания стали абсолютно неадекватными (отрицательный R² означает, что модель работает хуже, чем простое среднее)

**Вывод:** 
- Полиномиальные признаки высокой степени без регуляризации → гарантированное переобучение
- Ridge/L2 регуляризация не спасла ситуацию (alpha=1.0 недостаточно для степени 10)
- L1-регуляризация (Lasso) немного помогла, но тоже недостаточно

---

### **4. Наивные базовые модели (Task 9)**

| Модель | Train R² | Test R² | Интерпретация |
|--------|----------|---------|---------------|
| **Naive mean** | 0.000 | -0.000 | По определению R² = 0 для константного предсказания |
| **Naive median** | -0.059 | -0.057 | Хуже, чем mean (медиана не минимизирует MSE) |

**Вывод:** 
- Все обученные модели (R² test ≈ 0.40-0.45) значительно превосходят наивные предсказания
- Это подтверждает, что признаки (`bathrooms`, `bedrooms`) содержат полезную информацию для предсказания цены

---

### **5. Лучшая модель (Task 10)**

#### **По метрике R² на тесте:**

| Место | Модель | Test R² | Test RMSE | Stability (train-test) |
|:-----:|:-------|--------:|----------:|-----------------------:|
| 1 | **Lasso MinMaxScaler** | **0.416** | 1214.93 | 0.163 |
| 2 | ElasticNet StandardScaler | 0.451 | 1178.67 | 0.092 |
| 3 | Lasso default | 0.405 | 1226.62 | 0.175 |

#### **По стабильности (минимальный разрыв train-test):**

| Модель | \|train_R² - test_R²\| | Вывод |
|:-------|:---------------------:|:------|
| **ElasticNet StandardScaler** | **0.092** | Лучшая обобщающая способность |
| Lasso MinMaxScaler | 0.163 | Хорошая стабильность |
| Linear Regression | 0.175 | Средне |

---

### **6. Общий вывод**

**Лучшая модель для данного датасета** - Lasso с MinMaxScaler (или ElasticNet с StandardScaler)

Почему?:    
1. Наилучший R² на тесте (0.416-0.451)
2. Хорошая стабильность (разрыв train-test < 0.17)
3. L1-регуляризация автоматически отбирает важные признаки
4. Устойчивость к мультиколлинеарности

Что работает плохо: 
- Полиномиальные признаки степени 10 (большое переобучение)
- ElasticNet с MinMaxScaler (R² test = 0.129 — хуже всех)
- Naive baseline (R² ≈ 0)

#### **Финализируем:**
1. Нормализация: для бинарных признаков + 2 числовых — не критична, но для Lasso/ElasticNet рекомендуется
2. Регуляризация: L1/L2 помогают контролировать переобучение, но при alpha=1.0 эффект умеренный
3. Полиномиальные признаки: требуют **очень тщательного** подбора степени (2-3) и сильной регуляризации
4. Простая линейная регрессия: уже даёт приемлемый результат (R² = 0.405), сложные модели улучшают незначительно